# Reproducing Girardeau, Inema & Buzsaki (2017) from dandiset 000061

Paper. Girardeau G, Inema I, Buzsaki G (2017). Reactivations of emotional memory in the
hippocampus-amygdala system during sleep. *Nature Neuroscience* 20, 1634-1642.
[doi:10.1038/nn.4637](https://doi.org/10.1038/nn.4637).

Claims. `claims/000061/10.1038-nn.4637.yaml`, written by a reader from the version of record
(main text, figure legends, Online Methods, reporting summary) and the Supplementary Information,
with the paper's code (github.com/buzsakilab/papers, GGirardeau-BLAHpcInteractions-Package, at
commit cdcec89f), and checked by a second reader. Its criteria fix every method; the shared
methods M1 to M8 and the open details O1 to O7 are in its opening comment. This attempt
(`000061-girardeau-2017-02`) replaces `000061-girardeau-2017-01`, whose claims were written from
the abstract alone.

Dandiset. [DANDI:000061](https://dandiarchive.org/dandiset/000061/0.250624.0426), version
0.250624.0426. The paper cites no DANDI version (it says the data will be on CRCNS). There are
two published versions, 0.210812.1517 and 0.250624.0426, with the same 40 files, and the draft
(modified 2026-10-05) holds the same 40 assets as 0.250624.0426; the data check prints this.
The most recent published version is pinned, as the protocol asks when the paper cites none: it
is the version the earlier attempt tested, no later published version exists, and the draft
differs from it only in metadata the analysis does not use (keywords, anatomy terms, ethics
approval, affiliations).

What the dandiset holds. 40 NWB files, one per recording day, from four rats (Rat08 to Rat11):
sorted units with a region label, sleep states, epochs, and event times of hippocampal ripples,
air puffs and rewards; some files also hold raw voltage, LFP and video frames. There is no
position tracking, and the putative cell types the paper uses (pyramidal cell or interneuron,
from monosynaptic connections and then k-means on the waveform) are not in it: five Rat08 files
have a `cell_type` column that marks only cells identified as excitatory or inhibitory. Every
claim on putative pyramidal cells is therefore blocked.

No random numbers are drawn anywhere in this notebook.

| Claim | Tested here |
|---|---|
| `unit-counts` | tested |
| `rats-learn-puff-location` | blocked (too large: no position; only raw video frames) |
| `each-rat-learns-puff-location` | blocked (too large, depends on `rats-learn-puff-location`) |
| `dz-speed-slower-post-run` | blocked (too large, depends on `rats-learn-puff-location`) |
| `puff-location-no-speed-bias` | blocked (too large, depends on `rats-learn-puff-location`) |
| `habituation-over-days` | tested (laps under O1 reading c) |
| `bla-pyr-sleep-firing-increase` | tested (identified cells, five Rat08 files) |
| `bla-int-sleep-firing` | tested (identified cells, five Rat08 files) |
| `hpc-bla-reactivation-post-sleep` | blocked (metadata insufficient: no putative cell types) |
| `reactivation-all-cell-pairs` | tested |
| `reactivation-decay` | blocked (metadata insufficient) |
| `no-rem-reactivation` | blocked (metadata insufficient); the all-cell variant is shown |
| `piriform-weaker-reactivation` | blocked (metadata insufficient) |
| `cen-no-reactivation` | tested |
| `intra-structure-reactivation` | blocked (metadata insufficient); the all-cell tests are shown |
| `bla-ripple-up-subgroup` | blocked (metadata insufficient); fractions over all BLA units shown |
| `bla-ripple-down-subgroup` | blocked (metadata insufficient) |
| `ripple-modulated-cells-larger-reactivation` | blocked (depends on `bla-ripple-up-subgroup`); the all-cell test is shown |
| `run-correlated-pair-fractions` | blocked (metadata insufficient) |
| `reactivation-involves-ripple-up-cells` | blocked (metadata insufficient) |
| `pair-group-ev` | blocked (metadata insufficient) |
| `contributing-up-cells-gain-increase` | blocked (metadata insufficient) |
| `ev-independent-of-firing-rate` | blocked (metadata insufficient) |
| `reactivation-peaks-at-ripples` | blocked (depends on `hpc-bla-reactivation-post-sleep`); the all-cell test is shown |
| `airpuff-trajectory-ripple-reactivation` | blocked (metadata insufficient) |
| `safe-trajectory-no-ripple-reactivation` | blocked (metadata insufficient) |
| `aversive-direction-stronger` | blocked (metadata insufficient) |
| `direction-all-cell-pairs` | blocked (depends on `aversive-direction-stronger`); its three tests are shown |
| `bla-rate-same-across-directions` | blocked (metadata insufficient); the all-cell test is shown |
| `ripple-rate-unchanged` | tested |
| `contributing-pairs-run-sleep-coupling` | blocked (metadata insufficient) |
| `contributing-quartiles-run-sleep-coupling` | blocked (metadata insufficient) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hpcbla

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

notebook_started = started = time.time()
sessions = sorted(hpcbla.load_all(), key=lambda s: s["path"])
print(f"loaded {len(sessions)} files from dandiset {hpcbla.DANDISET_ID} version "
      f"{hpcbla.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output. The status of each
finding of the earlier attempt (`000061-girardeau-2017-01`) is summarised at the end of this
section.

In [ ]:
from dandi.dandiapi import DandiAPIClient

with DandiAPIClient() as client:
    ds = client.get_dandiset(hpcbla.DANDISET_ID)
    print("Versions:", [(v.identifier, str(v.modified)[:19]) for v in ds.get_versions()])
    meta = client.get_dandiset(hpcbla.DANDISET_ID, hpcbla.DANDISET_VERSION).get_raw_metadata()
    draft_meta = client.get_dandiset(hpcbla.DANDISET_ID, "draft").get_raw_metadata()
    pinned_assets = {a.path: a.identifier for a in client.get_dandiset(hpcbla.DANDISET_ID, hpcbla.DANDISET_VERSION).get_assets()}
    draft_assets = {a.path: a.identifier for a in client.get_dandiset(hpcbla.DANDISET_ID, "draft").get_assets()}
summary = meta["assetsSummary"]
print("Archive summary: files", summary["numberOfFiles"], "; subjects", summary["numberOfSubjects"],
      "; species", [s["name"] for s in summary["species"]], "; measured", summary["variableMeasured"])
print("Draft against pinned version: assets", len(draft_assets), "and", len(pinned_assets),
      "; same asset identifiers:", sum(pinned_assets.get(k) == v for k, v in draft_assets.items()))
skip = {"@context", "citation", "id", "url", "version", "doi", "datePublished", "publishedBy", "manifestLocation",
        "schemaVersion", "dateCreated", "dateModified", "assetsSummary"}
print("Metadata fields that differ in the draft:", sorted(k for k in set(meta) | set(draft_meta)
                                                     if k not in skip and meta.get(k) != draft_meta.get(k)))
print("Related resources (pinned):", [(r.get("relation"), r.get("identifier")) for r in meta.get("relatedResource", [])])

rows = []
for s in sessions:
    u = s["units"]
    pre, post, run = hpcbla.sleep_epochs(s)
    region = pd.Series(u["region"]).value_counts() if u else pd.Series(dtype=int)
    rows.append({
        "session": s["session"], "rat": s["rat"], "units": 0 if u is None else len(u["region"]),
        "hpc": int(sum(region.get(k, 0) for k in hpcbla.HPC_LABELS)),
        "bla": int(sum(region.get(k, 0) for k in hpcbla.BLA_LABELS)),
        "cen": int(sum(region.get(k, 0) for k in hpcbla.CEN_LABELS)),
        "pir": int(sum(region.get(k, 0) for k in hpcbla.PIR_LABELS)),
        "cell_type": bool(u and "cell_type" in u["columns"]),
        "states": s["states"] is not None,
        "ripples": 0 if s["ripples"] is None else len(s["ripples"]["peak"]),
        "puffs": len(s["events"]["puf"][1]) if "puf" in s["events"] else 0,
        "rewards": len(s["events"]["lrw"][1]) + len(s["events"]["rrw"][1]),
        "pre_sleep_s": round(float(np.diff(pre).sum())), "post_sleep_s": round(float(np.diff(post).sum())),
        "run_s": None if run is None else round(run[1] - run[0]),
        "lfp": s["lfp"] is not None, "raw": s["raw"] is not None, "videos": len(s["videos"]),
        "trials": None if s["trials"] is None else len(s["trials"][0])})
info = pd.DataFrame(rows)
print("\nSubjects:", sorted(info.rat.unique()), "; files per rat:", info.rat.value_counts().sort_index().to_dict())
print("Species, age, sex:", sorted({(s["species"], s["age"], s["sex"]) for s in sessions}))
print("\n", info.to_string(index=False))
print("\nFiles without a units table:", info.session[info.units == 0].tolist())
print("Files without sleep states:", info.session[~info.states].tolist())
print("Files without ripple events:", info.session[info.ripples == 0].tolist())
print("Rats with hippocampal units:", sorted(info.rat[info.hpc > 0].unique()))
print("Files with raw voltage:", int(info.raw.sum()), "; with LFP:", int(info.lfp.sum()), "; with video:", int((info.videos > 0).sum()))

The archive's summary says four subjects, which matches the four rats in the files, all male Long
Evans rats given as 3 months old. The draft holds the same assets as the pinned version (same
identifiers) and differs in descriptive metadata only, so no finding of the earlier attempt can
have been fixed in the data. Seven files have no units table, sixteen have no sleep states and
fourteen have no ripple events; the Rat09 files have no hippocampal units and no ripples, which
fits the paper's rat whose hippocampal probe failed (rat 2). The dandiset holds 40 recording
days, where the paper reports 61 sessions.

In [ ]:
labels = pd.Series(np.concatenate([s["units"]["label"] for s in sessions if s["units"]])).value_counts()
print("Unit location labels over all files (as stored):")
print(labels.to_string())
print("\nElectrode table locations:", sorted({x for s in sessions for x in s["electrode_locations"]}))
print("Files whose units table has no location column:",
      [s["session"] for s in sessions if s["units"] and not s["units"]["has_location_column"]])
print("\nUnits table columns:", sorted({tuple(sorted(s["units"]["columns"])) for s in sessions if s["units"]}))
typed = [s for s in sessions if s["units"] and "cell_type" in s["units"]["columns"]]
print("Files with a cell_type column:", [s["session"] for s in typed])
print("Its description:", sorted({s["units"]["cell_type_description"] for s in typed}))
ct = pd.DataFrame([{"session": s["session"], "region": s["units"]["region"][i], "cell_type": s["units"]["cell_type"][i],
                    "rate_hz": s["units"]["n_spikes"][i] / s["units"]["last_spike"]}
                   for s in typed for i in range(len(s["units"]["region"]))])
print(ct.groupby(["region", "cell_type"]).size().unstack(fill_value=0).to_string())
print("\nWhole-recording rate (Hz) by cell_type, all regions: median",
      ct.groupby("cell_type").rate_hz.median().round(2).to_dict())
ident = ct[ct.region.isin(hpcbla.BLA_LABELS)].groupby(["session", "cell_type"]).size().unstack(fill_value=0)
print("BLA units by cell_type and file:\n", ident.to_string())
hpc_exc = ct[ct.region.isin(hpcbla.HPC_LABELS) & (ct.cell_type == "excitatory")].groupby("session").size()
print("Hippocampal units typed excitatory per file:", hpc_exc.to_dict(), "; files with 6 or more:", int((hpc_exc >= 6).sum()))
print("\nUsed as hippocampus:", sorted(hpcbla.HPC_LABELS), "; BLA:", sorted(hpcbla.BLA_LABELS), "; central nuclei:",
      sorted(hpcbla.CEN_LABELS), "; piriform:", sorted(hpcbla.PIR_LABELS), "; wider amygdala variant:",
      sorted(hpcbla.BLA_COMPLEX_LABELS), "(case ignored)")

The unit labels are short codes with no description in the files or the dandiset metadata. They
match the structure names of the paper's Supplementary Fig. 3 (BLA, BLV, LaDL, BMP; Pir, VEn,
DEn; and CeN for the central nuclei), so `bla`, `cen`, `pir` and `hpc`/`HIP` are read as the
paper's BLA, central nuclei, piriform cortex and hippocampus (M1); this is consistent with the
paper's names, not shown by the dandiset. Some labels appear in two cases (`bla` and `BLA`,
`pir` and `PIR`, `plc01` and `PLCo1`); others (`cpu`, `no label`) are not defined anywhere. The
electrode table gives every channel the location `unknown`, and Rat09-20140328 has no location
column.

Cell types. Only five Rat08 files (20130709 to 20130713) have a `cell_type` column, described only
as "Type of cell this has been classified as", with the values excitatory, inhibitory and
unclassified. Most units are unclassified, and almost no hippocampal unit is typed (no file has six
typed excitatory hippocampal units), while the paper's k-means step types every remaining cell as
a putative pyramidal cell or interneuron. The column therefore holds at most the paper's
monosynaptic identification, and the putative types are missing: excitatory units are read as
monosynaptically identified pyramidal cells and inhibitory ones as identified interneurons (O4
reading a). This reading is consistent with the data (inhibitory units fire faster than
excitatory ones, and the BLA ratio of excitatory to inhibitory units is close to the paper's 675
to 175), not shown by it. The files hold no spike waveforms; the raw voltage of 16 files would be
needed to classify cells, which the claims file does not provide for. The earlier attempt did not
read this column. Every claim whose criterion uses putative pyramidal cells or interneurons is
blocked as `metadata_insufficient`.

In [ ]:
rows = []
for s in sessions:
    end_raw = max(b for _, _, b in s["epochs_raw"])
    lfp_s = s["lfp"][0][0] / s["lfp"][1] if s["lfp"] else np.nan
    rows.append({"session": s["session"], "epochs_last_stop_raw": end_raw,
                 "lfp_duration_s": round(lfp_s, 1), "ratio": round(end_raw / lfp_s, 3),
                 "last_spike_s": round(s["units"]["last_spike"], 1) if s["units"] else np.nan,
                 "lfp_rate_hz": s["lfp"][1] if s["lfp"] else np.nan,
                 "states_span_s": (round(min(a for _, a, _ in s["states"])), round(max(b for _, _, b in s["states"])))
                 if s["states"] else None,
                 "state_labels": ",".join(sorted({lab for lab, _, _ in s["states"]})) if s["states"] else None})
times = pd.DataFrame(rows)
print("Epoch table column description:", sorted({s["epoch_unit_stated"] for s in sessions}))
print(times.to_string(index=False))
print("\nRatio of the epoch table's last stop to the LFP duration: min", times.ratio.min(), "max", times.ratio.max())
print("Epoch tags:", sorted({t for s in sessions for t, _, _ in s["epochs"]}))
print("Files that tag sleep only as 'sleep':", {s["session"]: [t for t, _, _ in s["epochs"]] for s in sessions
                                               if "presleep" not in [t for t, _, _ in s["epochs"]] and "postsleep" not in [t for t, _, _ in s["epochs"]]})
print("Files with more than one run epoch:", {s["session"]: [(round(a), round(b)) for t, a, b in s["epochs"] if t == "run"]
                                             for s in sessions if sum(t == "run" for t, _, _ in s["epochs"]) > 1})

The epoch table says its times are in seconds, but its last stop time is 1000 times the LFP
duration in every file: the epochs are stored in milliseconds. The helper divides them by 1000.
Sleep states, spikes, ripples, puffs and rewards are in seconds and span the session; the states
(Awake, Non-REM, REM) cover the whole recording, so wake includes the track epochs. Three files
tag their sleep epochs only as `sleep`; there the sleep before the first run is pre-sleep and the
sleep after it post-sleep. Rat08-20130708 has a second run of about 600 s after its post-sleep;
the first run is the training run used here.

In [ ]:
rows = []
for s in sessions:
    eps = {t: (a, b) for t, a, b in s["epochs"]}
    for name, (n_frames, first, last, unit) in s["videos"].items():
        tag = name.rsplit("-", 1)[-1]
        a, b = eps.get(tag, (np.nan, np.nan))
        rows.append({"session": s["session"], "video": tag, "frames": n_frames, "stated_unit": unit,
                     "first_ts": round(first, 1), "last_ts": round(last, 1),
                     "video_span_ts/1000": round((last - first) / 1000, 1), "epoch_start_s": round(a, 1),
                     "epoch_duration_s": round(b - a, 1)})
videos = pd.DataFrame(rows)
print(videos.head(12).to_string(index=False))
print("...", len(videos), "videos in all")
ok = np.isfinite(videos.epoch_start_s)
print("Videos whose first timestamp / 1000 is within 5 s of their epoch's start:",
      int((np.abs(videos.first_ts[ok] / 1000 - videos.epoch_start_s[ok]) < 5).sum()), "of", int(ok.sum()))
print("Videos whose span / 1000 is within 5% of their epoch's duration:",
      int((np.abs(videos["video_span_ts/1000"][ok] / videos.epoch_duration_s[ok] - 1) < 0.05).sum()), "of", int(ok.sum()))
rows = []
for s in sessions:
    if s["trials"] is None:
        continue
    a, b, desc = s["trials"]
    _, _, run = hpcbla.sleep_epochs(s)
    rows.append({"session": s["session"], "trials": len(a), "description": desc,
                 "median_duration_s": round(float(np.median(b - a)), 1),
                 "fraction_in_run": round(float(np.mean((a >= run[0]) & (a < run[1]))), 2)})
print("\nTrials tables:")
print(pd.DataFrame(rows).to_string(index=False))

The video timestamps are stated in seconds but are in milliseconds, and they are not on the
session clock: the first timestamp matches the start of the epoch a video is named after in 1 of
114 videos, while the spans match the epochs' durations in 93 of 114. The videos are not used.
The `trials` table of the Rat08 and Rat09 files is described only as "experimental trials"; its
intervals last about 3 to 5 s and fall partly outside the run, so they are not laps (O1 reading
a has no lap intervals to use).

In [ ]:
# Video storage: frame shape and compression of every video series, the size of the run videos,
# and the time taken to stream frames of one run video.
storage = pd.DataFrame([{"session": s["session"], "video": k.rsplit("-", 1)[-1], "frames": shape[0],
                         "frame_shape": shape[1:], "dtype": dt, "compression": comp,
                         "uncompressed_gb": np.prod(shape) / 1e9}
                        for s in sessions for k, (shape, dt, comp) in s["video_storage"].items()])
print("Video series:", len(storage), "; frame shapes:", sorted(set(storage.frame_shape)), "; dtypes:", sorted(set(storage.dtype)),
      "; compression:", sorted(set(map(str, storage.compression))))
run_videos = storage[storage.video == "run"]
print("Run videos:", len(run_videos), "in", run_videos.session.nunique(), "files;", int(run_videos.frames.sum()), "frames,",
      round(float(run_videos.uncompressed_gb.sum())), "GB uncompressed")
first = sessions[0]
name = next(k for k in first["video_storage"] if k.endswith("-run"))
seconds, frame_bytes = hpcbla.time_frame_reads(dict(hpcbla.asset_urls())[first["path"]], name, 20)
print(f"Streaming 20 frames of {name}: {seconds:.1f} s ({seconds / 20:.2f} s per frame of {frame_bytes} bytes)")
print(f"At that rate the run frames alone would take about {run_videos.frames.sum() * seconds / 20 / 3600:.0f} hours to stream")
raw = [(s["session"], s["raw"]) for s in sessions if s["raw"]]
print("Raw voltage: files", len(raw), "; shapes and chunks, first three:", raw[:3])

The videos are stored as raw frames (360 by 640 pixels, three colours) inside the NWB files,
compressed with gzip, with no position derived from them. Tracking the rat in the run videos would
mean streaming the frames counted above and running a tracker, which does not fit the time limit,
and their timestamps would first have to be aligned to the session clock. The behaviour claims that
need position are blocked as too large. The raw voltage of 16 files is chunked one channel at a
time over about four hours of samples, so reading spike waveforms from it would mean streaming
whole channels.

In [ ]:
rows = []
for s in sessions:
    u = s["units"]
    if u and u["unsorted"]:
        rows.append({"session": s["session"], "unsorted_units": u["unsorted"],
                     "falls_back_after_spike": [x for i in u["unsorted"] for x in u["unsorted_at"][i]],
                     "unit0_last_spike_s": round(u["unit0_last"], 2), "unit1_first_spike_s": round(u["unit1_first"], 2),
                     "units_left_out": u["mixed"]})
print(pd.DataFrame(rows).to_string(index=False))
print(hpcbla.MIXED_UNITS_NOTE)

In ten files the spike times of unit 1 are not sorted: they fall back once, always after a
multiple of 65536 spikes, and unit 0 stops part way through the session at the time where unit 1
starts. The first part of unit 1 is the end of unit 0, so the index that separates them is wrong.
Units 0 and 1 of these files are left out of the analyses (they are still counted in
`unit-counts`, which counts units as stored).

In [ ]:
# Clock checks. (1) Ripples against the sleep states, with the states shifted by 600 s as the
# contrast. (2) Hippocampal multi-unit rate within 25 ms of ripple peaks against 1 s away.
rows = []
for s in sessions:
    if s["states"] is None or s["ripples"] is None or s["units"] is None:
        continue
    peaks = s["ripples"]["peak"]
    r = {"session": s["session"], "ripples_ordered": s["ripples"]["ordered"],
         "channels": ",".join(s["ripples"]["channels"]),
         "ripple_ms_median": round(1000 * float(np.median(s["ripples"]["stop"] - s["ripples"]["start"])), 1)}
    for label in ("Non-REM", "REM", "Awake"):
        iv = hpcbla.intervals_of(s, label)
        dur = np.diff(iv).sum()
        r[f"{label}_hz"] = round(np.count_nonzero(hpcbla.in_intervals(peaks, iv)) / dur, 3)
        r[f"{label}_hz_shift"] = round(np.count_nonzero(hpcbla.in_intervals(peaks, iv + 600)) / dur, 3)
    u = s["units"]
    hpc = hpcbla.units_of(s, hpcbla.HPC_LABELS)
    if hpc:
        spikes = np.sort(np.concatenate([u["trains"][i] for i in hpc]))
        def rate(t, w=0.025):
            return (np.searchsorted(spikes, t + w) - np.searchsorted(spikes, t - w)).mean() / (2 * w)
        r["hpc_hz_at_peak"] = round(rate(peaks), 1)
        r["hpc_hz_peak+1s"] = round(rate(peaks + 1), 1)
    rows.append(r)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))
print("\nRipples per s of Non-REM: min", clock["Non-REM_hz"].min(), "max", clock["Non-REM_hz"].max(), "; REM max", clock["REM_hz"].max())
print("Median over files: ripples per s in Non-REM", clock["Non-REM_hz"].median(),
      "(states shifted:", clock["Non-REM_hz_shift"].median(), "); in REM", clock["REM_hz"].median(),
      "(shifted:", clock["REM_hz_shift"].median(), ")")
print("Median ratio of hippocampal rate at ripple peaks to 1 s later:",
      round(float((clock.hpc_hz_at_peak / clock["hpc_hz_peak+1s"]).median()), 2), "; files with hippocampal units:",
      int(clock.hpc_hz_at_peak.notna().sum()))

In every file with states and ripples, ripples occur far more often per second of Non-REM than of
REM, and with the states shifted by 600 s the two rates move toward each other. Hippocampal units
fire several times faster within 25 ms of a ripple peak than 1 s later. The ripple annotations
come in ordered start, peak and stop triplets on one channel per file. So the states, the ripples
and the spikes share one clock, as far as these contrasts show.

In [ ]:
# (3) Lap direction decoded from hippocampal rates in each lap (leave one lap out, nearest
# centroid), with the true laps and with laps shifted by half their own duration.
def decode_direction(s, half_shift=False):
    u = s["units"]
    ids = hpcbla.units_of(s, hpcbla.HPC_LABELS)
    laps = hpcbla.run_laps(s)
    if len(ids) < 5 or len(laps) < 6 or len({d for _, _, d in laps}) < 2:
        return None
    if half_shift:
        laps = [(a + (b - a) / 2, b + (b - a) / 2, d) for a, b, d in laps]
    x = np.array([[np.count_nonzero((u["trains"][i] >= a) & (u["trains"][i] < b)) / (b - a) for i in ids]
                  for a, b, _ in laps])
    y = np.array([d == "LR" for _, _, d in laps])
    x = np.log1p(x)
    x = (x - x.mean(0)) / (x.std(0) + 1e-9)
    hits = 0
    for k in range(len(y)):
        keep = np.arange(len(y)) != k
        c1, c0 = x[keep & y].mean(0), x[keep & ~y].mean(0)
        hits += (np.sum((x[k] - c1) ** 2) < np.sum((x[k] - c0) ** 2)) == y[k]
    return hits / len(y), len(y)

rows = []
for s in sessions:
    if s["units"] is None:
        continue
    true, shifted = decode_direction(s), decode_direction(s, True)
    if true:
        rows.append({"session": s["session"], "laps": true[1], "accuracy": round(true[0], 3),
                     "accuracy_half_lap_shift": round(shifted[0], 3)})
dec = pd.DataFrame(rows)
print(dec.to_string(index=False))
print("Median accuracy", dec.accuracy.median(), "; with laps shifted by half a lap", dec.accuracy_half_lap_shift.median(),
      "; chance 0.5")

Hippocampal rates tell the lap direction apart in most laps built from the reward times, and less
well when the laps are shifted by half their length. The reward events and the spikes share one
clock, as far as this contrast shows; it does not test the timing to better than a fraction of a
lap.

In [ ]:
# (4) Air puffs: pooled rate in the 0.5 s after each puff against the 0.5 s before, in BLA
# (wider labels) and hippocampus, with the puff times shifted by 30 s as the contrast; and the lap
# direction that holds the puffs.
rows = []
for s in sessions:
    u = s["units"]
    if "puf" not in s["events"]:
        continue
    puffs = s["events"]["puf"][1]
    direction, counts = hpcbla.puff_direction(s)
    row = {"session": s["session"], "puffs": len(puffs), "direction": direction, **counts}
    for name, labels_ in (("bla", hpcbla.BLA_COMPLEX_LABELS), ("hpc", hpcbla.HPC_LABELS)):
        ids = hpcbla.units_of(s, labels_)
        if len(ids) < 3 or len(puffs) < 5:
            continue
        spikes = np.sort(np.concatenate([u["trains"][i] for i in ids]))
        def change(t):
            after = (np.searchsorted(spikes, t + 0.5) - np.searchsorted(spikes, t)).sum()
            before = (np.searchsorted(spikes, t) - np.searchsorted(spikes, t - 0.5)).sum()
            return (after - before) / max(before, 1)
        row[f"{name}_change"] = round(change(puffs), 3)
        row[f"{name}_change_shift30"] = round(change(puffs + 30), 3)
    rows.append(row)
puff = pd.DataFrame(rows)
print(puff.to_string(index=False))
print("\nMedian relative change after puffs, BLA:", puff.bla_change.median(), "(shifted:", puff.bla_change_shift30.median(),
      "); hippocampus:", puff.hpc_change.median(), "(shifted:", puff.hpc_change_shift30.median(), ")")
print("Files with BLA change above 0.3:", int((puff.bla_change > 0.3).sum()), "; with shifted times:",
      int((puff.bla_change_shift30 > 0.3).sum()))
has_dir = puff.direction.notna()
in_dir = int(sum(r[r.direction] for _, r in puff[has_dir].iterrows()))
print("Files with puffs:", len(puff), "; without a puff direction:", puff.session[puff.direction.isna()].tolist())
print("Puffs inside run laps of the air puff direction:", in_dir, "of", int(puff.puffs.sum()),
      "; in laps of the other direction:", int(sum(r["RL" if r.direction == "LR" else "LR"] for _, r in puff[has_dir].iterrows())),
      "; outside run laps:", int(puff.none.sum()))

In the 38 files with puffs, 1153 of 1197 puffs fall inside run laps of the direction that holds
most of them, 28 in laps of the other direction and 16 outside run laps. This gives the day's air
puff direction for the lap-based analyses (O1 reading c laps, M8 lap types). Rat08-20130709 has no
puff inside a run lap (2 puffs, 5 laps) and so no direction; two files have no puffs. The pooled response to puffs is small in most files;
this is consistent with the puffs sharing the spike clock but does not show it for every file.

In [ ]:
# Recordings deposited twice: compare unit spike counts and first spikes, ripple times and puff
# times between every pair of files, and units within a file.
def signature(s):
    u = s["units"]
    return {"units": None if u is None else tuple(zip(u["n_spikes"].tolist(), np.round(u["first_spike"], 4).tolist())),
            "ripples": None if s["ripples"] is None else tuple(np.round(s["ripples"]["peak"][:50], 4)),
            "puffs": tuple(np.round(s["events"]["puf"][1], 4)) if "puf" in s["events"] else None,
            "rewards": tuple(np.round(np.sort(np.concatenate([s["events"]["lrw"][1], s["events"]["rrw"][1]])), 4))}
sigs = {s["session"]: signature(s) for s in sessions}
names = list(sigs)
same = [(a, b, k) for i, a in enumerate(names) for b in names[i + 1:] for k in sigs[a]
        if sigs[a][k] is not None and sigs[a][k] == sigs[b][k]]
print("Pairs of files sharing units, ripples, puffs or rewards:", same if same else "none")
within = {s["session"]: len(sigs[s["session"]]["units"]) - len(set(sigs[s["session"]]["units"]))
          for s in sessions if s["units"]}
print("Units sharing spike count and first spike within a file:", {k: v for k, v in within.items() if v} or "none")

No two files share their units, ripples, puffs or rewards, and no two units of a file share a
spike count and first spike, so no recording was deposited twice. Units are not matched across
days; like the paper, each session is analysed on its own, and a cell recorded on several days may
be counted more than once.

In [ ]:
# Sessions entering each analysis: states, pre- and post-sleep, a run, and at least 6 cells in each
# structure (M4, M7); REM needs more than 180 s in both sleep epochs (M4).
rows = []
for s in sessions:
    if s["units"] is None:
        continue
    pre, post, run = hpcbla.sleep_epochs(s)
    has_sleep = s["states"] is not None and len(pre) > 0 and len(post) > 0 and run is not None
    r = {"session": s["session"], "rat": s["rat"], "sleep": has_sleep, "ripples": s["ripples"] is not None,
         "hpc": len(hpcbla.units_of(s, hpcbla.HPC_LABELS)), "bla": len(hpcbla.units_of(s, hpcbla.BLA_LABELS)),
         "cen": len(hpcbla.units_of(s, hpcbla.CEN_LABELS)),
         "bla_wide": len(hpcbla.units_of(s, hpcbla.BLA_COMPLEX_LABELS))}
    if has_sleep:
        rem = hpcbla.session_epoch_rows(s, "REM")
        nrem = hpcbla.session_epoch_rows(s)
        r.update({"pre_nrem_s": round(nrem["pre_s"]), "post_nrem_s": round(nrem["post_s"]),
                  "pre_rem_s": round(rem["pre_s"]), "post_rem_s": round(rem["post_s"]),
                  "rem_ok": rem["pre_s"] > hpcbla.MIN_REM_S and rem["post_s"] > hpcbla.MIN_REM_S})
    rows.append(r)
use = pd.DataFrame(rows)
use["hpc_bla"] = use.sleep & (use.hpc >= hpcbla.MIN_CELLS) & (use.bla >= hpcbla.MIN_CELLS)
use["hpc_cen"] = use.sleep & (use.hpc >= hpcbla.MIN_CELLS) & (use.cen >= hpcbla.MIN_CELLS)
print(use.to_string(index=False))
for col in ("hpc_bla", "hpc_cen"):
    d = use[use[col]]
    print(f"{col}: {len(d)} sessions {d.rat.value_counts().sort_index().to_dict()}; with REM above 180 s in both: {int(d.rem_ok.sum())}")
print("Units entering hpc_bla (hippocampus, BLA):", int(use.hpc[use.hpc_bla].sum()), int(use.bla[use.hpc_bla].sum()))

Under the claims file's rule (pre- and post-sleep NREM, a run, at least six cells of each
structure), the sessions and units printed above enter the hippocampus-BLA and
hippocampus-central nuclei analyses. Rat09 has no hippocampal units, so as in the paper only three
rats enter these analyses. Which of the paper's rats 1 to 4 each of Rat08 to Rat11 is, the
dandiset does not say; Rat09 is the rat without hippocampal units, as the paper's rat 2, and the
others are taken in order (Rat08 rat 1, Rat10 rat 3, Rat11 rat 4) for `unit-counts`, as an
assumption.

Status of the findings of `000061-girardeau-2017-01`. The version tested is the same
(0.250624.0426), and the draft holds the same assets, so none of the data findings can have been
fixed; each check above was re-run and gives the same result.

| Earlier finding | Status | Shown by |
|---|---|---|
| Epochs table in milliseconds though it says seconds | still present | epoch check (ratio 1000 in every file) |
| Video timestamps in milliseconds, stated as seconds, not on the session clock | still present | video check (1 of 114 starts match) |
| `spike_times_index` wrong at units 0 and 1 of ten files | still present | unsorted-unit check (ten files) |
| Files without units (7), sleep states (16) or ripples (14); Rat09 without hippocampal units | still present | file table |
| No position tracking, only raw video frames | still present | video storage check |
| Undescribed unit region labels; electrode table all `unknown`; one file without labels | still present in the dandiset; the paper's Supplementary Fig. 3 now gives the structure names the codes match | label check |
| `trials` table undescribed and not laps | still present | trials check |
| Ripple labels carry an undescribed number; events stored as stimulus annotations | still present | clock check (one channel per file) |
| Methods unavailable (paper not open, no code linked) | fixed for this attempt: the full text was supplied and the paper links its code; the dandiset still links no code (its related resources name only the paper) | header, related resources above |

New in this attempt: the `cell_type` column of five files, and the missing putative cell types
and waveforms (label check above).

## unit-counts

> Over 61 sessions in 4 rats, 7,390 well-isolated units were recorded (rat 1, 2,444; rat 2,
> 1,294; rat 3, 1,138; rat 4, 2,514), of which 2,038 were in the BLA, 782 in the central nuclei,
> 1,560 in the piriform cortex and 1,210 in the hippocampus.

Units as stored in the units tables, counted per rat and per location label (M1, case ignored).
Rats are matched to the paper's numbering as stated in the data check.

In [ ]:
rat_number = {"Rat08": "rat 1", "Rat09": "rat 2", "Rat10": "rat 3", "Rat11": "rat 4"}
units_all = pd.DataFrame([{"session": s["session"], "rat": s["rat"], "region": r}
                          for s in sessions if s["units"] for r in s["units"]["region"]])
region_sets = {"BLA": hpcbla.BLA_LABELS, "central nuclei": hpcbla.CEN_LABELS, "piriform cortex": hpcbla.PIR_LABELS,
               "hippocampus": hpcbla.HPC_LABELS}
expected_counts = [("sessions", 61, len(sessions)), ("units, all rats", 7390, len(units_all))]
for rat, paper_n in (("Rat08", 2444), ("Rat09", 1294), ("Rat10", 1138), ("Rat11", 2514)):
    expected_counts.append((f"units, {rat_number[rat]} ({rat})", paper_n, int((units_all.rat == rat).sum())))
for name, paper_n in (("BLA", 2038), ("central nuclei", 782), ("piriform cortex", 1560), ("hippocampus", 1210)):
    expected_counts.append((f"{name} units", paper_n, int(units_all.region.isin(region_sets[name]).sum())))
counts = pd.DataFrame(expected_counts, columns=["quantity", "paper", "dandiset"])
counts["tolerance"] = 0.1 * counts.paper
counts["within"] = (counts.dandiset - counts.paper).abs() <= counts.tolerance
print(counts.to_string(index=False))
print("\nFiles with a units table:", units_all.session.nunique(), "of", len(sessions))
print("Units by rat and region:")
print(units_all.assign(region=units_all.region.map(lambda r: next((k for k, v in region_sets.items() if r in v), "other: " + r)))
      .groupby(["region", "rat"]).size().unstack(fill_value=0).to_string())
print("\nVariant, wider amygdala labels (bla, blv, bmp, ladl):", int(units_all.region.isin(hpcbla.BLA_COMPLEX_LABELS).sum()))

**Result: not reproduced.** The dandiset holds 40 sessions (33 with a units table) and 4702
units, against the paper's 61 sessions and 7,390 units, both outside the 10% tolerance. Only
Rat08, taken as rat 1, is within tolerance (2317 units against 2,444); Rat09, Rat10 and Rat11 hold
515, 440 and 1430 units against 1,294, 1,138 and 2,514. By label the dandiset has 1118 BLA, 482
central nuclei, 935 piriform and 829 hippocampal units, against 2,038, 782, 1,560 and 1,210; with
the wider amygdala labels it has 1977 amygdala units. The dandiset holds part of the paper's data.**

## rats-learn-puff-location, each-rat-learns-puff-location, dz-speed-slower-post-run, puff-location-no-speed-bias (blocked)

> Rats learned the daily location of the air puff. The speed ratio between the previous and the
> current danger zone rises from pre-run to run and post-run, reversing between pre-run and
> post-run.

The four behaviour claims need the rat's speed in the 20 cm before the air puff location of the
day and of the previous day, which needs position on the track and the location of each puff.
The dandiset holds no position tracking and no puff location, only the puff times and the raw
video frames, and tracking the frames does not fit the time limit (data check: run frames and
streaming time). `each-rat-learns-puff-location`, `dz-speed-slower-post-run` and
`puff-location-no-speed-bias` depend on `rats-learn-puff-location` and are blocked with it.

**Result: blocked (too large).** No analysis is run.

## habituation-over-days

> The aversive valence of the air puff diminishes over training days; rats run more laps per unit
> time as they habituate.

Habituation index per session = laps / run epoch duration x 100 (Performance.m). Laps are O1
reading c (between consecutive rewards at opposite ends, both inside the run): readings a (lap
intervals the dataset holds) and b (position) cannot be run, as the dandiset has neither. The
three lap counts of the criterion: all laps (back-and-forth), air puff laps, and left-to-right
laps. Readings of "gradually diminished": (a) pooled Spearman correlation of day rank within the
rat and index, positive with p < 0.05; (b) positive Spearman correlation in at least 3 of the 4
rats (shown as the sign alone and with p < 0.05).

In [ ]:
rows = []
for s in sessions:
    _, _, run = hpcbla.sleep_epochs(s)
    laps_ = hpcbla.run_laps(s)
    direction, _ = hpcbla.puff_direction(s)
    duration = run[1] - run[0]
    rows.append({"session": s["session"], "rat": s["rat"], "run_s": round(duration), "laps": len(laps_),
                 "back_and_forth": 100 * len(laps_) / duration,
                 "airpuff": 100 * sum(d == direction for _, _, d in laps_) / duration if direction else np.nan,
                 "left_to_right": 100 * sum(d == "LR" for _, _, d in laps_) / duration})
hab = pd.DataFrame(rows)
hab["day"] = hab.groupby("rat").session.rank().astype(int)
print(hab.round(3).to_string(index=False))
rows = []
for kind in ("back_and_forth", "airpuff", "left_to_right"):
    d = hab.dropna(subset=[kind])
    rho, p = stats.spearmanr(d.day, d[kind])
    per_rat = {rat: stats.spearmanr(g.day, g[kind]) for rat, g in d.groupby("rat")}
    rows.append({"laps": kind, "sessions": len(d), "pooled_rho": round(rho, 3), "pooled_p": p,
                 "reading_a": bool(rho > 0 and p < 0.05),
                 **{f"rho_{rat}": round(r.statistic, 2) for rat, r in per_rat.items()},
                 **{f"p_{rat}": round(r.pvalue, 4) for rat, r in per_rat.items()},
                 "rats_positive": sum(r.statistic > 0 for r in per_rat.values()),
                 "rats_positive_p05": sum(r.statistic > 0 and r.pvalue < 0.05 for r in per_rat.values())})
hab_tests = pd.DataFrame(rows)
hab_tests["reading_b"] = hab_tests.rats_positive >= 3
print(hab_tests.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
for rat, d in hab.groupby("rat"):
    ax.plot(d.day, d.back_and_forth, "o-", label=rat)
ax.set_xlabel("training day within rat")
ax.set_ylabel("laps per 100 s of run")
ax.legend(fontsize=8)
plt.show()
print("Plotted range:", round(hab.back_and_forth.min(), 3), "to", round(hab.back_and_forth.max(), 3), "; no point outside the axes")

**Result: reproduced**, under O1 reading c, the only lap reading the dandiset allows. The
habituation index rises with training day for each lap count: pooled Spearman rho 0.383 (p =
0.015, 40 sessions) for all laps, 0.331 (p = 0.046, 37 sessions) for air puff laps and 0.397 (p =
0.011, 40 sessions) for left-to-right laps. The correlation is positive in all four rats for each
count, with p < 0.05 in Rat08, Rat09 and Rat10 and not in Rat11 (rho 0.32 to 0.33, p of 0.35 to
0.41). Both readings of "gradually diminished" hold for all three lap counts; the air puff count is
the closest to its threshold (p = 0.046).**

## bla-pyr-sleep-firing-increase

> BLA pyramidal cells fire faster in REM sleep and, to a lesser extent, in NREM sleep than in
> wakefulness.

Per BLA cell, rates over all REM, all NREM and all wake intervals of the session
(StatesFiringRates.m), ratios (state - wake)/(state + wake), and a two-sided Wilcoxon signed rank
test of each ratio against 0. Cells per O4 reading a: the dandiset's identified excitatory BLA
units (five Rat08 files); reading b (all putative pyramidal cells) cannot be run, as the putative
types are not in the dandiset.

In [ ]:
rows = []
for s in typed:
    for ctype in ("excitatory", "inhibitory"):
        for r in hpcbla.state_rates(s, hpcbla.units_of(s, hpcbla.BLA_LABELS, ctype)):
            rows.append({"session": s["session"], "type": ctype, **r})
state_fr = pd.DataFrame(rows)
for k in ("rem", "nrem"):
    state_fr[f"{k}_ratio"] = (state_fr[k] - state_fr.wake) / (state_fr[k] + state_fr.wake)

def ratio_tests(d):
    out = {}
    for k in ("rem", "nrem"):
        x = d[f"{k}_ratio"].dropna()
        out.update({f"{k}_n": len(x), f"{k}_median": round(float(x.median()), 4),
                    f"{k}_positive": int((x > 0).sum()), f"{k}_p": stats.wilcoxon(x).pvalue if len(x) > 1 else np.nan})
    return pd.Series(out)

print("Cells with an undefined ratio (no spike in either state):",
      state_fr.groupby("type")[["rem_ratio", "nrem_ratio"]].apply(lambda d: d.isna().sum()).to_dict())
print("\nIdentified BLA pyramidal cells (excitatory):")
pyr = state_fr[state_fr.type == "excitatory"]
print(ratio_tests(pyr).to_string())
print("\nBy session:")
print(pyr.groupby("session").apply(ratio_tests).to_string())
print("\nMedian rates (Hz): REM", round(pyr.rem.median(), 3), "; NREM", round(pyr.nrem.median(), 3), "; wake", round(pyr.wake.median(), 3))

**Result: reproduced**, on the 152 identified excitatory BLA units of five Rat08 sessions (O4
reading a; the paper had 675 cells from four rats). The REM/wake ratio has a median of 0.39
(positive in 120 of 152, p = 7.0e-15) and the NREM/wake ratio a median of 0.18 (positive in 108,
p = 7.2e-12), so both are positive and the REM median exceeds the NREM one. By session the REM
ratio is positive with p < 0.05 in four of five sessions and the NREM ratio in three;
Rat08-20130713 (22 cells) reaches neither.**

## bla-int-sleep-firing

> BLA interneurons do not change their firing rate between REM sleep and wake, and fire slightly
> less in NREM sleep than in wake.

As above, for the identified inhibitory BLA units (O4 reading a). The REM part is a claim of
absence, judged on the two-sided Wilcoxon signed rank test at p < 0.05.

In [ ]:
inter = state_fr[state_fr.type == "inhibitory"]
print("Identified BLA interneurons (inhibitory):")
print(ratio_tests(inter).to_string())
print("\nBy session:")
print(inter.groupby("session").apply(ratio_tests).to_string())
print("\nMedian rates (Hz): REM", round(inter.rem.median(), 3), "; NREM", round(inter.nrem.median(), 3), "; wake", round(inter.wake.median(), 3))

**Result: not reproduced.** On the 36 identified inhibitory BLA units of five Rat08 sessions,
the REM/wake ratio has a median of 0.029, positive in 22 of 36, with p = 0.020: the paper's test
finds a change between REM and wake, where the claim says there is none. The NREM/wake ratio has
a negative median (-0.035, positive in 16 of 36) but p = 0.26, so the decrease is not significant.
Neither part of the criterion holds. The cells are about a fifth of the paper's 175, from one rat,
and by session only Rat08-20130711 (12 cells) has a REM ratio with p < 0.05.**

## hpc-bla-reactivation-post-sleep (blocked)

> Hippocampus-BLA pyramidal cell pair correlations of the run are reinstated in post-sleep NREM,
> measured as explained variance (EV) exceeding the reverse explained variance (REV) across
> sessions.

The criterion uses putative pyramidal cells of the hippocampus and the BLA. The dandiset has no
putative types, and even its identified excitatory units leave no session with six hippocampal
cells (data check). The all-cell version of this test is the claim `reactivation-all-cell-pairs`,
below. `reactivation-decay` (pyramidal pairs, depends on this claim) and
`piriform-weaker-reactivation` (pyramidal pairs) are blocked for the same reason.

**Result: blocked (metadata insufficient).** No analysis is run here.

## reactivation-all-cell-pairs

> Hippocampus-BLA NREM reactivation holds when pyramidal cells and interneurons are both included.

EV and REV per session (M4) over all hippocampus-BLA pairs, from 50 ms bins of the run epoch and
of the NREM of pre- and post-sleep; two-sided Wilcoxon signed rank test of EV against REV across
sessions. The successive 20-min periods of post-NREM (24,000 bins each, with the whole pre-NREM
and run) are reported, as the criterion says, and decide nothing.

In [ ]:
started = time.time()
ev_rows = []
for s in sessions:
    r = use[use.session == s["session"]]
    if r.empty or not r.sleep.iloc[0]:
        continue
    hpc = hpcbla.units_of(s, hpcbla.HPC_LABELS)
    nrem_rows, rem_rows = hpcbla.session_epoch_rows(s), hpcbla.session_epoch_rows(s, "REM")
    row = {"session": s["session"], "rat": s["rat"]}
    for name, labels_ in (("bla", hpcbla.BLA_LABELS), ("cen", hpcbla.CEN_LABELS), ("bla_wide", hpcbla.BLA_COMPLEX_LABELS)):
        other = hpcbla.units_of(s, labels_)
        if len(hpc) < hpcbla.MIN_CELLS or len(other) < hpcbla.MIN_CELLS:
            continue
        res = hpcbla.ev_session(s, hpc, other, nrem_rows, post_periods=(name == "bla"))
        row.update({f"{name}_{k}": v for k, v in res.items()})
        if bool(r.rem_ok.iloc[0]) and name != "bla_wide":
            res = hpcbla.ev_session(s, hpc, other, rem_rows)
            row.update({f"{name}_rem_{k}": v for k, v in res.items()})
    ev_rows.append(row)
ev = pd.DataFrame(ev_rows)
print(f"computed in {time.time() - started:.0f} s")

def ev_test(df, prefix, label):
    d = df.dropna(subset=[f"{prefix}_ev"])
    diff = d[f"{prefix}_ev"] - d[f"{prefix}_rev"]
    res = stats.wilcoxon(d[f"{prefix}_ev"], d[f"{prefix}_rev"])
    print(f"{label}: {len(d)} sessions {d.rat.value_counts().sort_index().to_dict()}; EV median {d[f'{prefix}_ev'].median():.3f}%,"
          f" REV median {d[f'{prefix}_rev'].median():.3f}%; EV - REV mean {diff.mean():.3f}% (s.e.m. {diff.sem():.3f}),"
          f" median {diff.median():.3f}%, EV > REV in {int((diff > 0).sum())}; Wilcoxon two-sided p = {res.pvalue:.2g}")
    for rat, g in d.groupby("rat"):
        x = g[f"{prefix}_ev"] - g[f"{prefix}_rev"]
        print(f"  {rat}: {len(g)} sessions, EV - REV median {x.median():.3f}%, EV > REV in {int((x > 0).sum())}")
    return d, diff

cols = ["session", "rat", "bla_ev", "bla_rev", "bla_pairs", "bla_ev_0", "bla_rev_0", "bla_ev_1", "bla_rev_1", "bla_ev_2", "bla_rev_2", "bla_bins_2"]
print(ev[cols].dropna(subset=["bla_ev"]).round(3).to_string(index=False))
d_all, diff_all = ev_test(ev, "bla", "Hippocampus-BLA, all cells, NREM")

In [ ]:
print("Reported, not deciding: successive 20-min periods of post-NREM (all sessions of the test above)")
for k in range(3):
    x = d_all[f"bla_ev_{k}"] - d_all[f"bla_rev_{k}"]
    ok = x.dropna()
    print(f"  period {k + 1}: {len(ok)} sessions, EV - REV mean {ok.mean():.3f}%, EV > REV in {int((ok > 0).sum())},"
          f" Wilcoxon p = {stats.wilcoxon(ok).pvalue:.2g}")
print("Sessions whose third period is shorter than 20 min:", int((d_all.bla_bins_2 < hpcbla.PERIOD_BINS).sum()),
      "; post-NREM shorter than 40 min (no third period):", int(d_all.bla_ev_2.isna().sum()))
_ = ev_test(ev, "bla_wide", "Variant, wider amygdala labels (bla, blv, bmp, ladl)")
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(d_all.bla_rev, d_all.bla_ev, s=16)
lim = [0, max(d_all.bla_ev.max(), d_all.bla_rev.max()) * 1.05]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlabel("REV (%)")
ax.set_ylabel("EV (%)")
plt.show()
print("Plotted ranges: EV", round(d_all.bla_ev.min(), 3), "to", round(d_all.bla_ev.max(), 3), "; REV",
      round(d_all.bla_rev.min(), 3), "to", round(d_all.bla_rev.max(), 3), "; no point outside the axes")

**Result: reproduced.** Over 15 sessions (9 Rat08, 3 Rat10, 3 Rat11; the paper had 27), EV
exceeds REV in 13, with a mean EV - REV of 3.48% (s.e.m. 0.99) and a two-sided Wilcoxon p of
0.00031. By rat, EV exceeds REV in all 9 Rat08 and all 3 Rat11 sessions, and in 1 of 3 Rat10
sessions (median -0.023%). The wider amygdala labels give the same result (22 sessions, 17 with
EV > REV, p = 0.0005). Reported only: EV - REV falls over the successive 20-min periods of
post-NREM (means 2.49, 1.27 and 0.52%; p = 0.018, 0.035 and 1).**

## no-rem-reactivation (blocked)

> Hippocampus-BLA and hippocampus-piriform cortex reactivations are not significant in REM sleep.

The criterion tests pyramidal pairs, for hippocampus-BLA and hippocampus-piriform, which needs
the putative types. The criterion's variant, hippocampus-BLA with all cells in REM (sessions with
more than 180 s of REM in each sleep epoch), is shown and decides nothing.

In [ ]:
_ = ev_test(ev, "bla_rem", "Variant, hippocampus-BLA, all cells, REM")

**Result: blocked (metadata insufficient).** The printed variant is a description only: with all
cells, hippocampus-BLA EV does not differ from REV in REM (14 sessions, EV > REV in 8, p = 0.43).

## cen-no-reactivation

> There are no reactivations between the central amygdala nuclei and the hippocampus.

A claim of absence, judged on the paper's test: EV and REV (M4) over all hippocampus-central
nuclei pairs, two-sided Wilcoxon signed rank of EV against REV in NREM and in REM (REM sessions
with more than 180 s in each sleep epoch).

In [ ]:
print(ev[["session", "rat", "cen_ev", "cen_rev", "cen_pairs", "cen_rem_ev", "cen_rem_rev"]].dropna(subset=["cen_ev"]).round(3).to_string(index=False))
d_cen, diff_cen = ev_test(ev, "cen", "Hippocampus-central nuclei, all cells, NREM")
d_cen_rem, diff_cen_rem = ev_test(ev, "cen_rem", "Hippocampus-central nuclei, all cells, REM")

**Result: reproduced.** Hippocampus-central nuclei EV does not differ from REV in NREM (9
sessions, median EV - REV 0.60%, EV > REV in 7, p = 0.30) or in REM (8 sessions, median 0.085%,
EV > REV in 6, p = 0.25), so the paper's test finds no reactivation. The paper had 19 sessions in
each test; with 9 and 8 the test has less power, and the NREM median is positive.**

## intra-structure-reactivation (blocked)

> Within the hippocampus and within the BLA, pair correlations are reactivated in NREM but not in
> REM sleep.

The criterion needs all four NREM tests, two of them on putative pyramidal pairs, so it cannot be
decided. The all-cell tests (distinct pairs within one structure, M4) are shown and decide nothing.

In [ ]:
rows = []
for s in sessions:
    r = use[use.session == s["session"]]
    if r.empty or not r.sleep.iloc[0]:
        continue
    for name, labels_ in (("hippocampus", hpcbla.HPC_LABELS), ("BLA", hpcbla.BLA_LABELS)):
        ids = hpcbla.units_of(s, labels_)
        if len(ids) < hpcbla.MIN_CELLS:
            continue
        row = {"session": s["session"], "rat": s["rat"], "structure": name, "cells": len(ids)}
        res = hpcbla.ev_session(s, ids, None, hpcbla.session_epoch_rows(s))
        row.update({"nrem_ev": res["ev"], "nrem_rev": res["rev"]})
        if bool(r.rem_ok.iloc[0]):
            res = hpcbla.ev_session(s, ids, None, hpcbla.session_epoch_rows(s, "REM"))
            row.update({"rem_ev": res["ev"], "rem_rev": res["rev"]})
        rows.append(row)
intra = pd.DataFrame(rows)
for name, g in intra.groupby("structure"):
    for state in ("nrem", "rem"):
        d = g.dropna(subset=[f"{state}_ev"])
        x = d[f"{state}_ev"] - d[f"{state}_rev"]
        print(f"{name}, all pairs, {state.upper()}: {len(d)} sessions {d.rat.value_counts().sort_index().to_dict()},"
              f" EV - REV median {x.median():.3f}%, EV > REV in {int((x > 0).sum())}, Wilcoxon p = {stats.wilcoxon(x).pvalue:.2g}")

**Result: blocked (metadata insufficient).** The printed tests are a description only: with all
pairs, the hippocampus reactivates in NREM (23 sessions, EV > REV in 19, p = 0.0054) and the BLA
does not reach p < 0.05 (16 sessions, 9, p = 0.14); in REM neither does (p = 0.83 and 0.21).

## bla-ripple-up-subgroup, bla-ripple-down-subgroup (blocked)

> A subset of BLA neurons is positively modulated during hippocampal SPW-Rs, 25.8% of
> interneurons (42 of 163) and 11.1% of pyramidal cells (137 of 1,233).

Both claims give fractions of putative pyramidal cells and interneurons, which the dandiset does
not type, and are blocked. The Poisson test of M6 (both readings of O2) is run on every BLA unit of
the sessions with states and ripples, and its fractions are shown over all BLA units and over the
identified units of the five typed files, as a description only. The ripple modulation is also
used in the description of `ripple-modulated-cells-larger-reactivation`, below.

In [ ]:
rows = []
for s in sessions:
    if s["units"] is None or s["states"] is None or s["ripples"] is None:
        continue
    for r in hpcbla.ripple_modulation(s, hpcbla.units_of(s, hpcbla.BLA_LABELS)):
        rows.append({"session": s["session"], "rat": s["rat"], "cell_type": s["units"]["cell_type"][r["unit"]] or "(no column)", **r})
mod = pd.DataFrame(rows)
print("BLA units tested:", len(mod), "in", mod.session.nunique(), "sessions; untested (no spike in ripples or baseline):", int((~mod.tested).sum()))
summary = mod.groupby("cell_type")[["up_a", "down_a", "up_b", "down_b"]].agg(["sum", "mean"]).round(3)
summary.insert(0, "units", mod.groupby("cell_type").size())
print(summary.to_string())
print("All BLA units: up", int(mod.up_a.sum()), f"({mod.up_a.mean():.3f}) reading a,", int(mod.up_b.sum()), f"({mod.up_b.mean():.3f}) reading b;",
      "down", int(mod.down_a.sum()), f"({mod.down_a.mean():.3f}) reading a,", int(mod.down_b.sum()), f"({mod.down_b.mean():.3f}) reading b")
print(mod.groupby("rat")[["up_a", "down_a"]].agg(["sum", "mean"]).round(3).to_string())

**Result: blocked (metadata insufficient).** `bla-ripple-down-subgroup` depends on this claim and
is blocked with it. The printed fractions are a description only: over all 822 BLA units of 17
sessions, 105 (12.8%) are upmodulated and 85 (10.3%) downmodulated under O2 reading a (110 and 91
under reading b); among the identified units of five files, 8 of 36 inhibitory (22.2%) and 9 of
152 excitatory units (5.9%) are upmodulated.

## ripple-modulated-cells-larger-reactivation (blocked)

> Reactivation (EV - REV) is larger for hippocampus-BLA pairs whose BLA cell is ripple-modulated
> than for pairs with a non-modulated BLA cell.

The claim depends on `bla-ripple-up-subgroup` and is blocked with it; its pyramidal test also needs
the putative types. Its all-cell test is shown as a description: per session, EV - REV with the
modulated BLA units (up or down, M6) and with the non-modulated ones, sessions with more than 5
hippocampal and more than 5 modulated BLA units; two-sided Wilcoxon rank sum test, and the signed
rank test on the same sessions as the criterion's variant.

In [ ]:
rows = []
for s in sessions:
    r = use[use.session == s["session"]]
    if r.empty or not r.sleep.iloc[0] or s["ripples"] is None:
        continue
    hpc = hpcbla.units_of(s, hpcbla.HPC_LABELS)
    m = mod[mod.session == s["session"]]
    for reading in ("a", "b"):
        is_mod = m[f"up_{reading}"] | m[f"down_{reading}"]
        modulated, other = [int(u) for u in m.unit[is_mod]], [int(u) for u in m.unit[~is_mod]]
        if len(hpc) <= 5 or len(modulated) <= 5:
            continue
        rows_ = hpcbla.session_epoch_rows(s)
        a = hpcbla.ev_session(s, hpc, modulated, rows_)
        b = hpcbla.ev_session(s, hpc, other, rows_)
        rows.append({"session": s["session"], "rat": s["rat"], "reading": reading, "modulated": len(modulated), "other": len(other),
                     "mod_ev_rev": a["ev"] - a["rev"], "non_ev_rev": b["ev"] - b["rev"]})
modev = pd.DataFrame(rows)
print(modev.round(3).to_string(index=False))
for reading, d in modev.groupby("reading"):
    d = d.dropna()
    print(f"O2 reading {reading}: {len(d)} sessions; EV - REV mean modulated {d.mod_ev_rev.mean():.2f}%, non-modulated {d.non_ev_rev.mean():.2f}%;"
          f" rank sum p = {stats.ranksums(d.mod_ev_rev, d.non_ev_rev).pvalue:.2g}; signed rank p = {stats.wilcoxon(d.mod_ev_rev, d.non_ev_rev).pvalue:.2g};"
          f" modulated larger in {int((d.mod_ev_rev > d.non_ev_rev).sum())}")

**Result: blocked (metadata insufficient, through `bla-ripple-up-subgroup`).** The printed tests
are a description only: with all cells, EV - REV is larger with the modulated BLA units in 7 of 12
sessions (means 4.18 and 1.68%), with rank sum p = 0.27 under O2 reading a and 0.3 under reading b.

## run-correlated-pair-fractions, reactivation-involves-ripple-up-cells, pair-group-ev, contributing-up-cells-gain-increase, ev-independent-of-firing-rate, contributing-pairs-run-sleep-coupling, contributing-quartiles-run-sleep-coupling (blocked)

> Among nine groups of hippocampus-BLA pyramidal pairs formed by run correlation and BLA ripple
> modulation, the pairs with a significant positive run correlation and an upmodulated BLA partner
> show the largest pre-NREM to post-NREM increase in correlation, and differ from every other group.

Each of these criteria pools hippocampus-BLA pyramidal pairs (putative types, M1), and all but
`ev-independent-of-firing-rate` also use the ripple modulation of `bla-ripple-up-subgroup`. The
dandiset has no putative types, so all seven are blocked. The earlier attempt tested
`reactivation-involves-ripple-up-cells` with joint-pattern weights over all BLA units, a measure the
claims file no longer uses.

**Result: blocked (metadata insufficient).** No analysis is run.

## reactivation-peaks-at-ripples (blocked)

> Reactivation strength of the whole-run template around hippocampal ripple peaks is higher in
> post-NREM than in pre-NREM, for pyramidal pairs and for all cell pairs.

The claim depends on `hpc-bla-reactivation-post-sleep`, which is blocked, and its first test uses
pyramidal cells. Its second test, with all cells, is shown as a description: R per M7 with the whole
run as template, the ripple-peak value of pre- and post-NREM per session, one-tailed Wilcoxon
signed rank test (post greater than pre). `airpuff-trajectory-ripple-reactivation`,
`safe-trajectory-no-ripple-reactivation` and `aversive-direction-stronger` test pyramidal cells and
are blocked; `direction-all-cell-pairs` repeats them with all cells and depends on
`aversive-direction-stronger`, so it is blocked too, and its three tests are shown below with the
air puff and safe templates (laps per O1 reading c).

In [ ]:
started = time.time()
rows, curves = [], {}
for s in sessions:
    r = use[use.session == s["session"]]
    if r.empty or not r.hpc_bla.iloc[0] or s["ripples"] is None:
        continue
    _, _, run = hpcbla.sleep_epochs(s)
    hpc, bla = hpcbla.units_of(s, hpcbla.HPC_LABELS), hpcbla.units_of(s, hpcbla.BLA_LABELS)
    for template, iv in (("run", np.array([run])), ("airpuff", hpcbla.lap_intervals(s, "puff")), ("safe", hpcbla.lap_intervals(s, "safe"))):
        if len(iv) == 0:
            continue
        out = hpcbla.reactivation_strength(s, hpc, bla, iv)
        curves[(s["session"], template)] = (out["pre_curve"], out["post_curve"])
        rows.append({"session": s["session"], "rat": s["rat"], "template": template,
                     **{k: v for k, v in out.items() if not k.endswith("curve")}})
rs = pd.DataFrame(rows)
rs["post_minus_pre"] = rs.post_peak - rs.pre_peak
print(f"computed in {time.time() - started:.0f} s")
print(rs.round(4).to_string(index=False))

def one_tailed(d, label):
    res = stats.wilcoxon(d.post_peak, d.pre_peak, alternative="greater")
    print(f"{label}: {len(d)} sessions {d.rat.value_counts().sort_index().to_dict()}; post minus pre median {d.post_minus_pre.median():.4f},"
          f" positive in {int((d.post_minus_pre > 0).sum())}; one-tailed Wilcoxon p = {res.pvalue:.2g}")

one_tailed(rs[rs.template == "run"], "Whole-run template, all cells")
centres = hpcbla.peri_centres()
mean_pre = np.mean([curves[k][0] for k in curves if k[1] == "run"], axis=0)
mean_post = np.mean([curves[k][1] for k in curves if k[1] == "run"], axis=0)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(centres, mean_pre, label="pre-NREM")
ax.plot(centres, mean_post, label="post-NREM")
ax.axvspan(-0.25, 0.25, color="0.9")
ax.set_xlabel("time from ripple peak (s)")
ax.set_ylabel("mean z-scored R, all cells")
ax.legend(fontsize=8)
plt.show()
print("Plotted range:", round(float(min(mean_pre.min(), mean_post.min())), 4), "to", round(float(max(mean_pre.max(), mean_post.max())), 4))

**Result: blocked (metadata insufficient, through `hpc-bla-reactivation-post-sleep`).** The
printed test is a description only: with all cells, the ripple-peak strength of the whole-run
template is higher in post-NREM than in pre-NREM in 11 of 15 sessions (median difference 0.0224,
one-tailed p = 0.0042).

In [ ]:
print("direction-all-cell-pairs, its three tests with all cells (description only):")
puff_r, safe_r = rs[rs.template == "airpuff"], rs[rs.template == "safe"]
one_tailed(puff_r, "Air puff template, post greater than pre")
one_tailed(safe_r, "Safe template, post greater than pre")
both = puff_r.set_index("session")[["post_minus_pre"]].join(safe_r.set_index("session")[["post_minus_pre"]], lsuffix="_airpuff", rsuffix="_safe").dropna()
res = stats.wilcoxon(both.post_minus_pre_airpuff, both.post_minus_pre_safe, alternative="greater")
print(f"Air puff greater than safe: {len(both)} sessions; median difference {(both.post_minus_pre_airpuff - both.post_minus_pre_safe).median():.4f},"
      f" positive in {int((both.post_minus_pre_airpuff > both.post_minus_pre_safe).sum())}; one-tailed Wilcoxon p = {res.pvalue:.2g}")
print("Template bins (air puff, safe):", {k: (int(g[g.template == 'airpuff'].template_bins.sum()), int(g[g.template == 'safe'].template_bins.sum()))
                                          for k, g in rs.groupby("session") if (g.template == "airpuff").any()})

**Result: blocked (metadata insufficient, through `aversive-direction-stronger`).** The printed
tests are a description only: with all cells over 14 sessions, the air puff template is more
reactivated in post-NREM (10 of 14, one-tailed p = 0.01), so is the safe template (12 of 14, p =
0.0043), and the air puff change does not exceed the safe one (8 of 14, p = 0.48), the pattern the
claim states. With all cells, `aversive-direction-stronger`'s comparison is not significant here.

## bla-rate-same-across-directions (blocked)

> Firing rates of BLA cells do not differ significantly between the safe and the air puff
> trajectories.

The criterion needs both the pyramidal-cell test and the all-cell test, so it cannot be decided.
The all-cell test is shown as a description: per BLA unit, the spike count over the air puff laps
and over the safe laps of the run divided by their durations (O7 reading a; reading b needs
position), two-sided Wilcoxon signed rank test, over the cells of the sessions of
`direction-all-cell-pairs` (reading a of the sessions) and over all BLA cells of sessions with a
run and a puff direction (reading b).

In [ ]:
rows = []
for s in sessions:
    if s["units"] is None or hpcbla.puff_direction(s)[0] is None:
        continue
    ivs = {k: hpcbla.lap_intervals(s, k) for k in ("puff", "safe")}
    if any(len(v) == 0 for v in ivs.values()):
        continue
    for i in hpcbla.units_of(s, hpcbla.BLA_LABELS):
        t = s["units"]["trains"][i]
        rows.append({"session": s["session"], "rat": s["rat"], "unit": i,
                     **{f"{k}_hz": hpcbla.count_in(t, iv) / np.diff(iv).sum() for k, iv in ivs.items()}})
dir_fr = pd.DataFrame(rows)
for label, d in (("sessions of direction-all-cell-pairs", dir_fr[dir_fr.session.isin(both.index)]), ("all sessions with a puff direction", dir_fr)):
    x = d.puff_hz - d.safe_hz
    print(f"{label}: {len(d)} BLA cells in {d.session.nunique()} sessions; median air puff minus safe {x.median():.4f} Hz,"
          f" Wilcoxon p = {stats.wilcoxon(d.puff_hz, d.safe_hz).pvalue:.2g}")

**Result: blocked (metadata insufficient).** The printed tests are a description only: with all
BLA cells, the rate on air puff laps exceeds that on safe laps by a median of 0.0036 Hz, p = 0.022
(739 cells, 14 sessions), and by 0.0032 Hz, p = 0.0048 (1033 cells, 21 sessions), so the all-cell
test would find a difference under both session readings.

## ripple-rate-unchanged

> The occurrence rate of SPW-Rs does not differ between pre-NREM and post-NREM.

A claim of absence, judged on the paper's test: per session with ripple events, ripple peaks per
second of pre-NREM and of post-NREM (RippleRatePrePost.m), two-sided Wilcoxon signed rank test.

In [ ]:
rows = []
for s in sessions:
    pre, post, run = hpcbla.sleep_epochs(s)
    if s["states"] is None or s["ripples"] is None or len(pre) == 0 or len(post) == 0:
        continue
    rows.append({"session": s["session"], "rat": s["rat"], **hpcbla.ripple_rates(s)})
rip_rate = pd.DataFrame(rows)
rip_rate["post_minus_pre"] = rip_rate.post_rate - rip_rate.pre_rate
print(rip_rate.round(3).to_string(index=False))
res = stats.wilcoxon(rip_rate.pre_rate, rip_rate.post_rate)
print(f"\n{len(rip_rate)} sessions {rip_rate.rat.value_counts().sort_index().to_dict()}; median pre {rip_rate.pre_rate.median():.3f} Hz,"
      f" post {rip_rate.post_rate.median():.3f} Hz; median post minus pre {rip_rate.post_minus_pre.median():.4f} Hz,"
      f" higher in post in {int((rip_rate.post_minus_pre > 0).sum())}; Wilcoxon two-sided p = {res.pvalue:.2g}")
for rat, g in rip_rate.groupby("rat"):
    print(f"  {rat}: {len(g)} sessions, median post minus pre {g.post_minus_pre.median():.4f} Hz, higher in post in {int((g.post_minus_pre > 0).sum())}")
print(f"\nNotebook run time: {(time.time() - notebook_started) / 60:.1f} min")

**Result: reproduced.** Over 23 sessions (the paper had 41), the median ripple rate is
0.611 Hz in pre-NREM and 0.587 Hz in post-NREM; post minus pre has a median of 0.0315 Hz, is
positive in 15, and the two-sided Wilcoxon p is 0.26. By rat the median change is 0.089 Hz in
Rat08 (higher in post in 8 of 9), -0.008 Hz in Rat10 (3 of 7) and 0.007 Hz in Rat11 (4 of 7).**

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome | Earlier attempt (-01) |
|---|---|---|
| `unit-counts` | not reproduced (40 of 61 sessions, 4702 of 7,390 units) | no claim |
| `rats-learn-puff-location` | blocked: too large (no position) | blocked: too large |
| `each-rat-learns-puff-location` | blocked: too large | no claim |
| `dz-speed-slower-post-run` | blocked: too large | no claim |
| `puff-location-no-speed-bias` | blocked: too large | no claim |
| `habituation-over-days` | reproduced (O1 reading c) | no claim |
| `bla-pyr-sleep-firing-increase` | reproduced (152 identified cells, one rat) | no claim |
| `bla-int-sleep-firing` | not reproduced (REM/wake p = 0.020; NREM p = 0.26) | no claim |
| `hpc-bla-reactivation-post-sleep` | blocked: metadata insufficient | reproduced (another measure) |
| `reactivation-all-cell-pairs` | reproduced (p = 0.00031, 15 sessions) | no claim |
| `reactivation-decay` | blocked: metadata insufficient | no claim |
| `no-rem-reactivation` | blocked: metadata insufficient | no claim |
| `piriform-weaker-reactivation` | blocked: metadata insufficient | no claim |
| `cen-no-reactivation` | reproduced (NREM p = 0.30, REM p = 0.25) | no claim |
| `intra-structure-reactivation` | blocked: metadata insufficient | no claim |
| `bla-ripple-up-subgroup` | blocked: metadata insufficient | reproduced (another test) |
| `bla-ripple-down-subgroup` | blocked: metadata insufficient | no claim |
| `ripple-modulated-cells-larger-reactivation` | blocked: metadata insufficient | no claim |
| `run-correlated-pair-fractions` | blocked: metadata insufficient | no claim |
| `reactivation-involves-ripple-up-cells` | blocked: metadata insufficient | reproduced (another measure) |
| `pair-group-ev` | blocked: metadata insufficient | no claim |
| `contributing-up-cells-gain-increase` | blocked: metadata insufficient | no claim |
| `ev-independent-of-firing-rate` | blocked: metadata insufficient | no claim |
| `reactivation-peaks-at-ripples` | blocked: metadata insufficient | reproduced (another measure) |
| `airpuff-trajectory-ripple-reactivation` | blocked: metadata insufficient | no claim |
| `safe-trajectory-no-ripple-reactivation` | blocked: metadata insufficient | no claim |
| `aversive-direction-stronger` | blocked: metadata insufficient | not reproduced (another measure) |
| `direction-all-cell-pairs` | blocked: metadata insufficient | no claim |
| `bla-rate-same-across-directions` | blocked: metadata insufficient | no claim |
| `ripple-rate-unchanged` | reproduced (p = 0.26, 23 sessions) | no claim |
| `contributing-pairs-run-sleep-coupling` | blocked: metadata insufficient | no claim |
| `contributing-quartiles-run-sleep-coupling` | blocked: metadata insufficient | no claim |

How the analysis differs from the paper's. Every method is the claims file's, from the paper's
Methods and code. The differences come from the data: the dandiset holds 40 of the paper's 61
sessions; it has no putative cell types, so the cells of `bla-pyr-sleep-firing-increase` and
`bla-int-sleep-firing` are the identified excitatory and inhibitory units of five Rat08 files (O4
reading a), and every claim on putative pyramidal cells is blocked; it has no position, so laps
are the intervals between rewards at opposite ends (O1 reading c) and the behaviour claims are
blocked; brain regions are the dandiset's labels read as the paper's structures; units 0 and 1 of
ten files are left out of the analyses because their spike times are mixed. Bins belong to an
interval when their centre does, and ripples to NREM when their peak does (standard choices).

What was found about the dandiset.
- The putative cell types the paper uses are missing; a `cell_type` column in five of 33 files
  with units marks only identified excitatory and inhibitory cells and is described in one line.
  There are no waveforms.
- It holds 40 recording days against the paper's 61, and 4702 units against 7,390.
- Still present from the earlier attempt (the files are unchanged): the epochs table is in
  milliseconds although it says seconds; video timestamps are in milliseconds, stated as seconds,
  and are not on the session clock; in ten files the units index is wrong at units 0 and 1; seven
  files have no units table, sixteen no sleep states and fourteen no ripple events; there is no
  position tracking; the unit region labels, the `trials` table and the ripple labels are
  undescribed.
- The dandiset links no analysis code; the paper links its code.